# Laboratorio 1 · Azure AI Document Intelligence
**Proyecto Integrador:** Copiloto Cognitivo de Asistencia Bancaria · **Participante:** AnitaCM7

Este notebook reproduce las pruebas de la sesión de nivelación (`nb_DocIntell.ipynb`) sobre documentos **100% sintéticos** del caso:

| Documento | Archivo | Origen | Modelo |
|---|---|---|---|
| Guía de procedimientos de atención de tarjetas de crédito `GUI-ATC-TC-007` de **Banco Nova Andino** (ficticio, 3 págs., 9 secciones, 8 tablas) | `data/guia_atencion_tarjetas_nova.pdf` | Generada con IA (Claude) para el proyecto | `prebuilt-layout` |
| Factura ficticia de **Andes Print Lab S.A.C.** a Banco Nova Andino (1 pág.) | `data/factura_sintetica_andes_print.pdf` | Generada con IA (Claude) para el proyecto | `prebuilt-invoice` |

La guía es coherente con la ficha de productos de Banco Nova Andino entregada por el instructor (mismos productos, ingresos mínimos y canales), y agrega el contenido operativo que consultaría el Copiloto: requisitos, flujo de evaluación, bloqueo, reclamos, escalamiento y refinanciamiento.

**Autenticación:** `DefaultAzureCredential` toma la identidad de `az login`. No se usan API Keys; el `.env` contiene solo endpoints.

## Parte A · Validación de conexión con Azure
- Azure CLI instalada y sesión iniciada con `az login` usando la cuenta asignada del laboratorio.
- `az account show` confirmó: suscripción **Azure subscription 1**, tenant **Data Scientist Cognitive NTT** (`DSCNTT.onmicrosoft.com`), `isDefault: true`, usuario de tipo `user`.
- Los notebooks se autentican con `DefaultAzureCredential` (identidad de `az login`). **No se usó ninguna API Key**; el `.env` contiene solo endpoints y nombres de recursos.
- El Paso 3 de este notebook lo confirma: el recurso respondió con 67 modelos sin errores 401/403.

## Paso 2 · Librerías y clientes con la identidad del participante

In [1]:
import os
import re
import json
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import HttpResponseError
from azure.ai.documentintelligence import (
    DocumentIntelligenceClient,
    DocumentIntelligenceAdministrationClient,
)
from azure.ai.documentintelligence.models import DocumentContentFormat

load_dotenv(override=True)  # busca el .env subiendo desde esta carpeta

endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
assert endpoint, "Falta DOCUMENT_INTELLIGENCE_ENDPOINT en el .env"

credential = DefaultAzureCredential()
client = DocumentIntelligenceClient(endpoint=endpoint, credential=credential)
admin_client = DocumentIntelligenceAdministrationClient(endpoint=endpoint, credential=credential)

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

DOC_PRINCIPAL = DATA_DIR / "guia_atencion_tarjetas_nova.pdf"
DOC_FACTURA = DATA_DIR / "factura_sintetica_andes_print.pdf"

print("Endpoint:", endpoint)
print("Guía existe:", DOC_PRINCIPAL.exists(), "| Factura existe:", DOC_FACTURA.exists())

Endpoint: https://docintellntt.cognitiveservices.azure.com/
Guía existe: True | Factura existe: True


## Paso 3 · Listar los modelos disponibles del recurso

In [2]:
try:
    modelos = list(admin_client.list_models())
    print("Autenticación correcta. Modelos disponibles:", len(modelos))
    for m in modelos:
        marca = "  <-- prebuilt" if m.model_id.startswith("prebuilt-") else ""
        print(f"- {m.model_id} | {m.description}{marca}")
except HttpResponseError as e:
    # Regla del lab: si aparece 401/403, registrar y avisar al instructor. No modificar permisos.
    print(f"Error HTTP {e.status_code}: {e.message}")

Autenticación correcta. Modelos disponibles: 67
- prebuilt-bankStatement.us | Extract key information from US bank statement.  <-- prebuilt
- prebuilt-check.us | Extract key information from United States check.  <-- prebuilt
- prebuilt-contract | Extract key information from legal contracts.  <-- prebuilt
- prebuilt-creditCard | Extract key information from credit card.  <-- prebuilt
- prebuilt-healthInsuranceCard.us | Extract key information from US health insurance cards.  <-- prebuilt
- prebuilt-idDocument | Extract key information from passports and ID cards.  <-- prebuilt
- prebuilt-invoice | Extract key information from invoices.  <-- prebuilt
- prebuilt-layout | Extract text and layout information from documents.  <-- prebuilt
- prebuilt-marriageCertificate.us | Extract key information from US marriage certificate.  <-- prebuilt
- prebuilt-mortgage.us.1003 | Extract key information from US Uniform Residential Loan Application (Form 1003).  <-- prebuilt
- prebuilt-mortgage.us.10

## Paso 4 · `prebuilt-layout` sobre la guía de atención de tarjetas
Se solicita la salida en **Markdown** (`output_content_format`): conserva encabezados y tablas, lo cual facilita el chunking por sección para AI Search en el Paso 6.

In [3]:
def analizar(ruta, model_id, **kwargs):
    with open(ruta, "rb") as f:
        poller = client.begin_analyze_document(
            model_id=model_id,
            body=f,
            content_type="application/octet-stream",
            **kwargs,
        )
    return poller.result()

layout = analizar(DOC_PRINCIPAL, "prebuilt-layout",
                  output_content_format=DocumentContentFormat.MARKDOWN)

print("Modelo utilizado:", layout.model_id)
print("Número de páginas:", len(layout.pages or []))
print("Párrafos detectados:", len(layout.paragraphs or []))
print("Tablas detectadas:", len(layout.tables or []))
print("Caracteres de contenido:", len(layout.content or ""))

Modelo utilizado: prebuilt-layout
Número de páginas: 3
Párrafos detectados: 175
Tablas detectadas: 8
Caracteres de contenido: 8802


### 4.1 Texto, roles y secciones detectadas

In [4]:
for p in layout.pages:
    print(f"Página {p.page_number}: {len(p.lines or [])} líneas | {p.width} x {p.height} {p.unit}")

print("\nTítulos y encabezados de sección (role):")
for par in layout.paragraphs or []:
    if par.role in ("title", "sectionHeading", "pageHeader", "pageFooter"):
        print(f"  [{par.role}] {par.content}")

Página 1: 83 líneas | 8.2639 x 11.6806 LengthUnit.INCH
Página 2: 84 líneas | 8.2639 x 11.6806 LengthUnit.INCH
Página 3: 45 líneas | 8.2639 x 11.6806 LengthUnit.INCH

Títulos y encabezados de sección (role):
  [ParagraphRole.PAGE_HEADER] BANCO NOVA ANDINO · GUI-ATC-TC-007 · Versión 1.0 · Uso interno
  [ParagraphRole.TITLE] Guía de procedimientos de atención de tarjetas de crédito Banco Nova Andino · Copiloto Cognitivo de Asistencia Bancaria
  [ParagraphRole.SECTION_HEADING] 1. Objetivo y alcance
  [ParagraphRole.SECTION_HEADING] 2. Requisitos para solicitar una tarjeta
  [ParagraphRole.SECTION_HEADING] 3. Flujo de evaluación de la solicitud
  [ParagraphRole.PAGE_FOOTER] Documento 100% sintético para el Laboratorio 1 · Data Scientist Cognitive
  [ParagraphRole.PAGE_HEADER] BANCO NOVA ANDINO · GUI-ATC-TC-007 · Versión 1.0 · Uso interno
  [ParagraphRole.SECTION_HEADING] 4. Bloqueo y reposición de tarjeta
  [ParagraphRole.SECTION_HEADING] 5. Reclamos y desconocimiento de consumos
  [Paragra

### 4.2 Tablas detectadas

In [5]:
def tabla_a_markdown(tabla):
    # Reconstruye la grilla fila/columna a partir de las celdas
    grilla = [["" for _ in range(tabla.column_count)] for _ in range(tabla.row_count)]
    for c in tabla.cells:
        grilla[c.row_index][c.column_index] = (c.content or "").replace("\n", " ").replace("|", "/")
    lineas = ["| " + " | ".join(grilla[0]) + " |",
              "|" + "---|" * tabla.column_count]
    lineas += ["| " + " | ".join(fila) + " |" for fila in grilla[1:]]
    return "\n".join(lineas)

for i, t in enumerate(layout.tables or [], start=1):
    pagina = t.bounding_regions[0].page_number if t.bounding_regions else "?"
    print(f"\nTabla {i} · página {pagina} · {t.row_count} filas x {t.column_count} columnas")
    print(tabla_a_markdown(t))


Tabla 1 · página 1 · 5 filas x 2 columnas
| Campo | Detalle |
|---|---|
| Código / vigencia | GUI-ATC-TC-007 · vigente desde 01/09/2026 |
| Área responsable | Gerencia de Experiencia del Cliente - Tarjetas |
| Aplica a | Especialistas de agencia, banca telefónica y Copiloto Cognitivo |
| Productos | Nova Clásica, Nova Oro, Nova Platinum y Nova Eco |

Tabla 2 · página 1 · 5 filas x 5 columnas
| Tarjeta | Ingreso mínimo | Antigüedad laboral | Sustento de ingresos | Calificación crediticia |
|---|---|---|---|---|
| Nova Clásica | S/ 1,800 | 6 meses | Última boleta o recibo por honorarios | Normal o CPP |
| Nova Oro | S/ 3,500 | 12 meses | Últimas 2 boletas | Normal |
| Nova Platinum | S/ 7,500 | 24 meses | Últimas 3 boletas o declaración anual | Normal (12 meses) |
| Nova Eco | S/ 2,800 | 6 meses | Última boleta o recibo por honorarios | Normal o CPP |

Tabla 3 · página 1 · 6 filas x 4 columnas
| Etapa | Responsable | Plazo máximo | Resultado |
|---|---|---|---|
| 1. Registro de solicitu

### 4.3 Contenido completo en Markdown (evidencia)

In [6]:
(OUT_DIR / "guia_atencion_tarjetas_layout.md").write_text(layout.content, encoding="utf-8")
print(layout.content[:3000])
print("\n... guardado completo en outputs/guia_atencion_tarjetas_layout.md")

<!-- PageHeader="BANCO NOVA ANDINO · GUI-ATC-TC-007 · Versión 1.0 · Uso interno" -->


# Guía de procedimientos de atención de tarjetas de crédito Banco Nova Andino · Copiloto Cognitivo de Asistencia Bancaria


<table>
<tr>
<th>Campo</th>
<th>Detalle</th>
</tr>
<tr>
<td>Código / vigencia</td>
<td>GUI-ATC-TC-007 · vigente desde 01/09/2026</td>
</tr>
<tr>
<td>Área responsable</td>
<td>Gerencia de Experiencia del Cliente - Tarjetas</td>
</tr>
<tr>
<td>Aplica a</td>
<td>Especialistas de agencia, banca telefónica y Copiloto Cognitivo</td>
</tr>
<tr>
<td>Productos</td>
<td>Nova Clásica, Nova Oro, Nova Platinum y Nova Eco</td>
</tr>
</table>


Documento ficticio creado con IA para fines educativos. El banco, sus procedimientos, plazos, costos y teléfonos no corresponden a una entidad
real.


## 1. Objetivo y alcance

Esta guía estandariza la atención de consultas y solicitudes sobre tarjetas de crédito. Define los requisitos de solicitud,
el flujo de evaluación, los plazos de bloqueo y reposi

**Observaciones:**
- Se detectaron 3 páginas, 175 párrafos y **las 8 tablas** del documento (3 por página en las páginas 1 y 2, y 2 en la página 3). La relación fila/columna se conservó: cada ingreso mínimo, antigüedad laboral y calificación quedó asociada a su tarjeta, y cada plazo a su etapa o tipo de reclamo.
- Se reconocieron correctamente el `title` y las **9 secciones numeradas** como `sectionHeading`, en el orden del documento.
- El encabezado y el pie que se repiten en cada página se identificaron como `pageHeader` y `pageFooter` (3 de cada uno), así que es fácil excluirlos al indexar.
- Las celdas con texto largo (por ejemplo, "5 días hábiles (Lima) / 10 (provincias)" o las acciones de escalamiento) se mantuvieron en una sola celda.
- El título y el subtítulo del documento se unieron en un solo párrafo `title` ("Guía de procedimientos... · Copiloto Cognitivo de Asistencia Bancaria").

## Paso 5 · `prebuilt-invoice` sobre la factura sintética

In [7]:
factura = analizar(DOC_FACTURA, "prebuilt-invoice")

print("Modelo utilizado:", factura.model_id)
print("Número de páginas:", len(factura.pages or []))
print("Tablas detectadas:", len(factura.tables or []))
print("Documentos (facturas) reconocidos:", len(factura.documents or []))

# Evidencia completa
with open(OUT_DIR / "factura_prebuilt_invoice_raw.json", "w", encoding="utf-8") as f:
    json.dump(factura.as_dict(), f, ensure_ascii=False, indent=2)

Modelo utilizado: prebuilt-invoice
Número de páginas: 1
Tablas detectadas: 3
Documentos (facturas) reconocidos: 1


In [8]:
def valor(campo):
    # Devuelve el valor tipado del campo; si no existe, su texto
    if campo is None:
        return None
    if campo.value_currency is not None:
        return f"{campo.value_currency.currency_code or ''} {campo.value_currency.amount}".strip()
    for attr in ("value_date", "value_string", "value_number", "value_integer"):
        v = getattr(campo, attr, None)
        if v is not None:
            return v
    if campo.value_address is not None:
        return campo.content
    return campo.content

for idx, doc in enumerate(factura.documents or [], start=1):
    print(f"-------- Factura #{idx} | tipo: {doc.doc_type} | confianza: {doc.confidence:.2f} --------")
    for nombre, campo in sorted(doc.fields.items()):
        if nombre == "Items":
            continue
        conf = f"{campo.confidence:.2f}" if campo.confidence is not None else "-"
        print(f"{nombre:<28} = {str(valor(campo)):<55} (conf {conf})")

    items = doc.fields.get("Items")
    if items and items.value_array:
        print(f"\nÍtems detectados: {len(items.value_array)}")
        for n, it in enumerate(items.value_array, start=1):
            o = it.value_object or {}
            print(f"  {n}. {valor(o.get('Description'))} | cant: {valor(o.get('Quantity'))} "
                  f"| p.unit: {valor(o.get('UnitPrice'))} | importe: {valor(o.get('Amount'))}")

-------- Factura #1 | tipo: invoice | confianza: 1.00 --------
AmountDue                    = PEN 53170.8                                             (conf 0.45)
CustomerAddress              = Jr. Las Begonias 1550, Piso 9, San Isidro, Lima - Perú  (conf 0.82)
CustomerAddressRecipient     = Banco Nova Andino S.A.
Atención: Gerencia de Tarjetas de Crédito (conf 0.69)
CustomerName                 = Banco Nova Andino S.A.
Atención: Gerencia de Tarjetas de Crédito (conf 0.69)
CustomerTaxId                = 20888777665                                             (conf 0.93)
DueDate                      = 2026-12-03                                              (conf 0.94)
InvoiceDate                  = 2026-11-03                                              (conf 0.94)
InvoiceId                    = F001-004827                                             (conf 0.96)
InvoiceTotal                 = PEN 53170.8                                             (conf 0.45)
PaymentTerm                 

**Valores esperados (ground truth de la factura sintética)** para comparar con la extracción:

| Campo | Valor esperado |
|---|---|
| VendorName | Andes Print Lab S.A.C. |
| CustomerName | Banco Nova Andino S.A. |
| InvoiceId | F001-004827 |
| InvoiceDate / DueDate | 03/11/2026 / 03/12/2026 |
| PurchaseOrder | OC-NA-2026-0391 |
| SubTotal / TotalTax / InvoiceTotal | S/ 45,060.00 / S/ 8,110.80 / S/ 53,170.80 |
| Items | 5 líneas |

Ojo: el formato `dd/mm/aaaa` puede interpretarse como `mm/dd`; revisar `InvoiceDate`.

In [9]:
esperado = {"InvoiceId": "F001-004827", "PurchaseOrder": "OC-NA-2026-0391",
            "SubTotal": 45060.00, "TotalTax": 8110.80, "InvoiceTotal": 53170.80}

doc = (factura.documents or [None])[0]
if doc:
    for k, v in esperado.items():
        c = doc.fields.get(k)
        obtenido = (c.value_currency.amount if c and c.value_currency else (c.content if c else None))
        ok = (abs(float(obtenido) - v) < 0.01) if isinstance(v, float) and obtenido is not None else (obtenido == v)
        print(f"{'OK ' if ok else 'REV'} {k:<14} esperado={v!s:<18} obtenido={obtenido}")

OK  InvoiceId      esperado=F001-004827        obtenido=F001-004827
OK  PurchaseOrder  esperado=OC-NA-2026-0391    obtenido=OC-NA-2026-0391
OK  SubTotal       esperado=45060.0            obtenido=45060.0
OK  TotalTax       esperado=8110.8             obtenido=8110.8
OK  InvoiceTotal   esperado=53170.8            obtenido=53170.8


**Observaciones:**
- Se reconoció 1 factura (confianza 1.00) y los 5 ítems con cantidad, precio unitario e importe correctos. Los 5 valores de control (InvoiceId, PurchaseOrder, SubTotal, TotalTax, InvoiceTotal) coinciden exactamente con la factura sintética.
- Las fechas en formato `dd/mm/aaaa` se interpretaron bien (03/11/2026 → 2026-11-03) y la moneda se normalizó a `PEN`. También extrajo los RUC como `VendorTaxId`/`CustomerTaxId`.
- **Confianza baja en InvoiceTotal y AmountDue (0.45)**, aunque el valor extraído es correcto. En un flujo productivo convendría un umbral (p. ej. < 0.70) que envíe el campo a revisión humana o lo valide con una regla de negocio (`SubTotal + TotalTax = InvoiceTotal`).
- `CustomerName` incluyó la línea "Atención: Gerencia de Tarjetas de Crédito": el modelo mezcla el destinatario con el área. `TaxDetails` no se pobló, porque el modelo no reconoce el desglose de IGV como detalle de impuesto.

## Paso 6 · JSON para indexar en Azure AI Search
Se usa la **guía de atención** (no la factura), porque es la fuente que consultará el Copiloto. Se divide **un registro por sección** (chunking por encabezado Markdown), conservando las tablas dentro de su sección.

El esquema replica exactamente los campos del índice de clase (`nb_AISearch.ipynb`): `id, titulo, contenido, categoria, area, fecha_publicacion, vigente, url_documento`. Así el archivo puede cargarse directamente con `search_client.upload_documents(...)`.

In [10]:
def chunks_por_seccion(markdown):
    # Divide el markdown en bloques que empiezan con un encabezado (#, ##, ###)
    bloques, actual, titulo = [], [], "Encabezado del documento"
    for linea in markdown.splitlines():
        m = re.match(r"^#{1,3}\s+(.*)", linea)
        if m and actual:
            bloques.append((titulo, "\n".join(actual).strip()))
            actual = []
        if m:
            titulo = m.group(1).strip()
        actual.append(linea)
    if actual:
        bloques.append((titulo, "\n".join(actual).strip()))
    return [(t, c) for t, c in bloques if len(c) > 40]

def chunks_por_pagina(resultado):
    # Respaldo si el layout no devolvió encabezados: un chunk por página
    salida = []
    for p in resultado.pages:
        texto = "".join(resultado.content[s.offset:s.offset + s.length] for s in p.spans)
        salida.append((f"Página {p.page_number}", texto.strip()))
    return salida

secciones = chunks_por_seccion(layout.content)
if len(secciones) < 3:
    print("Pocos encabezados detectados; se usa chunking por página.")
    secciones = chunks_por_pagina(layout)

fecha = datetime.now(timezone.utc).strftime("%Y-%m-%dT00:00:00Z")
documentos = []
for i, (titulo, contenido) in enumerate(secciones, start=1):
    documentos.append({
        "id": f"nova-guia-{i:02d}",                     # solo letras, números, guiones
        "titulo": f"Guía de atención de tarjetas - {titulo}",
        "contenido": contenido,
        "categoria": "Procedimientos de atención",
        "area": "Experiencia del Cliente - Tarjetas",
        "fecha_publicacion": fecha,
        "vigente": True,
        "url_documento": "https://novabanco.example/docs/GUI-ATC-TC-007.pdf",
    })

ruta_json = OUT_DIR / "nova_andino_guia_index.json"
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(documentos, f, ensure_ascii=False, indent=2)

print(f"{len(documentos)} registros guardados en {ruta_json}")
for d in documentos:
    print(f"- {d['id']} | {d['titulo'][:70]} | {len(d['contenido'])} chars")

11 registros guardados en outputs/nova_andino_guia_index.json
- nova-guia-01 | Guía de atención de tarjetas - Encabezado del documento | 84 chars
- nova-guia-02 | Guía de atención de tarjetas - Guía de procedimientos de atención de t | 706 chars
- nova-guia-03 | Guía de atención de tarjetas - 1. Objetivo y alcance | 451 chars
- nova-guia-04 | Guía de atención de tarjetas - 2. Requisitos para solicitar una tarjet | 1116 chars
- nova-guia-05 | Guía de atención de tarjetas - 3. Flujo de evaluación de la solicitud | 1077 chars
- nova-guia-06 | Guía de atención de tarjetas - 4. Bloqueo y reposición de tarjeta | 865 chars
- nova-guia-07 | Guía de atención de tarjetas - 5. Reclamos y desconocimiento de consum | 989 chars
- nova-guia-08 | Guía de atención de tarjetas - 6. Límites del Copiloto y reglas de esc | 1421 chars
- nova-guia-09 | Guía de atención de tarjetas - 7. Refinanciamiento para clientes con d | 752 chars
- nova-guia-10 | Guía de atención de tarjetas - 8. Ejemplo resuelto (fecha 

**Observaciones:** se generaron **11 registros**, uno por cada sección numerada (9), más el título con la tabla de datos generales (`nova-guia-02`) y el encabezado de página (`nova-guia-01`). A diferencia de la ficha, aquí **ninguna sección se perdió**: todas tienen contenido suficiente y cada tabla viaja dentro de su sección (por ejemplo, los requisitos por tarjeta en `nova-guia-04` y las reglas de escalamiento en `nova-guia-08`). El registro `nova-guia-01` (84 caracteres) contiene solo el comentario `PageHeader`, que es ruido y debe excluirse antes de indexar en AI Search, filtrando los roles `pageHeader`/`pageFooter`.

## Aplicación al Proyecto Integrador
**¿Los documentos que utilizaría el Copiloto contienen tablas, formularios, documentos escaneados u otros elementos que justifiquen Azure AI Document Intelligence?**

Sí. Las fichas de producto, tarifarios y procedimientos que consultaría un especialista bancario son mayormente **tabulares y condicionales**. En la guía de atención, la información clave está en tablas: requisitos por tarjeta (ingreso, antigüedad, sustento, calificación), plazos del flujo de evaluación, costos de reposición, plazos de reclamos y reglas de escalamiento. La ficha de productos, por su parte, compara membresías, TCEA y consumos para exonerar. Un extractor de texto plano rompe la relación fila/columna. Entonces el Copiloto podría atribuir el requisito o el plazo de una fila a otra (por ejemplo, la antigüedad laboral de la Nova Oro a la Nova Eco). `prebuilt-layout` conserva esa estructura y además identifica encabezados de sección, lo que permite un chunking semántico por sección.

**Información que se espera extraer:** requisitos por producto, plazos y responsables de cada procedimiento, reglas condicionales (abono provisional menor a S/ 2,000, escalamiento por tipo de situación), notas al pie con exclusiones, y metadatos (tipo de documento, fecha, vigencia). `prebuilt-invoice` no es central para el Copiloto, pero sirve si se incorporan comprobantes o documentos sustentatorios de clientes en flujos de evaluación.

## Reflexión
- **Capacidades útiles:** extracción de tablas con estructura de celdas, roles de párrafo (`title`, `sectionHeading`), salida en Markdown lista para RAG y campos tipados con confianza en `prebuilt-invoice`.
- **Cómo aplicarlas:** pipeline de ingesta *PDF → prebuilt-layout (Markdown) → chunking por sección → JSON → AI Search*. Las tablas viajan dentro de su chunk, así el LLM recibe la comparación completa.
- **Por validar:** (1) calidad en documentos escaneados o de baja resolución; (2) tablas que cruzan páginas; (3) tamaño óptimo de chunk y si conviene un chunk por fila de producto; (4) formato de fechas `dd/mm` y montos en soles en `prebuilt-invoice`; (5) uso de un umbral de confianza para mandar campos a revisión humana.